# 3. ROMs and other output-only nodes

Some layers have **only outputs** — they are free-running sources in RTL, but
the **emulator** still needs a packet count. That count comes from a packet
source (`pkt_src`) registered on the DAG.

Covered here:

1. `Rom` + consumer with a real activation input as `pkt_src`
2. A DAG with **no activation inputs**, paced by a **pseudo-input**


In [1]:
from pathlib import Path

import numpy as np
from fpga_ai_suite.spatial.flow import run_flow
from fpga_ai_suite.spatial.layers import Add, Constant, Directed_acyclic_graph, Rom

WORK = Path("work_roms")
WORK.mkdir(exist_ok=True)


## `Rom` + `Add` with a real input as `pkt_src`

`Rom` / `Constant` / `Range` expose `dout` only. Call
`register_rom_dependency(rom=..., pkt_src=..., pkt_multiplier=1)` so the
emulator knows how many packets to emit.

`pkt_src` must be a port that exists in this DAG (usually a top-level input).


In [2]:
class Rom_add_dag(Directed_acyclic_graph):
    """
    din --------> Add ---> dout
                   ^
                   |
    Rom -----------+
    """

    def __init__(self, weights, num_lanes=4, dw=8):
        self.weights_data = weights
        self.num_lanes = num_lanes
        self.dw = dw
        super().__init__(module_name="Rom_add_dag")

    def init_topology(self):
        self.input = self.create_input("din", self.dw, self.num_lanes, "INT")
        self.output = self.create_output("dout", self.dw + 1, self.num_lanes, "INT")

        self.rom = self.add_node(
            Rom(
                weights=self.weights_data,
                num_lanes=self.num_lanes,
                dw=self.dw,
                dtype="INT",
                mif_init=True,
            )
        )
        self.adder = self.add_node(Add(num_lanes=self.num_lanes, dw=self.dw, dtype="INT"))

        # On each forward call, Rom will generate the same amount of packets as self.input
        self.register_rom_dependency(rom=self.rom, pkt_src=self.input, pkt_multiplier=1)

        self.connect(self.input, self.adder.inputs.dina)
        self.connect(self.rom.outputs.dout, self.adder.inputs.dinb)
        self.connect(self.adder.outputs.dout, self.output)


rom_weights = np.arange(8, dtype=np.int32).reshape(1, 8)
rom_model = Rom_add_dag(rom_weights)
din = np.ones((3, 1, 8), dtype=np.int32)
out, = rom_model(din=din)
print("Rom+Add out shape:", out.shape, "sample:", out[0])


Rom+Add out shape: (3, 1, 8) sample: [[1 2 3 4 5 6 7 8]]


## Pseudo-input when the DAG has no activation inputs

If the functional data path is only free-running sources (`Rom` / `Constant`),
you still need a port in the DAG to act as `pkt_src`:

1. `create_input(...)` — a **pseudo-input** used to determine packet count.
2. `register_rom_dependency(..., pkt_src=pseudo_input)` for every Rom/Constant
3. Feed dummy packets on that port when you call `forward` / `set_shape`

The pseudo-input does **not** have to sit on the activation data path. Below,
`pkt_src` is only used for `Rom` and `Constant`; the Add consumes those two sources.


In [3]:
class Rom_pseudo_input_dag(Directed_acyclic_graph):
    """No activation data input."""

    def __init__(self, weights, num_lanes=4, dw=8):
        self.weights_data = weights
        self.num_lanes = num_lanes
        self.dw = dw
        super().__init__(module_name="Rom_pseudo_input_dag")

    def init_topology(self):
        # Pseudo packet source: exists so register_rom_dependency has a port in-DAG
        self.pkt_src = self.create_input("pkt_src", self.dw, self.num_lanes, "INT")
        self.output = self.create_output("dout", self.dw + 1, self.num_lanes, "INT")

        self.rom = self.add_node(
            Rom(
                weights=self.weights_data,
                num_lanes=self.num_lanes,
                dw=self.dw,
                dtype="INT",
                mif_init=True,
            )
        )
        self.zero = self.add_node(
            Constant(value=0, m=8, n=1, num_lanes=self.num_lanes, dw=self.dw, dtype="INT")
        )
        self.adder = self.add_node(Add(num_lanes=self.num_lanes, dw=self.dw, dtype="INT"))

        self.register_rom_dependency(rom=self.rom, pkt_src=self.pkt_src, pkt_multiplier=1)
        self.register_rom_dependency(rom=self.zero, pkt_src=self.pkt_src, pkt_multiplier=1)

        self.connect(self.rom.outputs.dout, self.adder.inputs.dina)
        self.connect(self.zero.outputs.dout, self.adder.inputs.dinb)
        self.connect(self.adder.outputs.dout, self.output)
        # Note: pkt_src is intentionally not connected on the data path


pseudo = Rom_pseudo_input_dag(rom_weights)
# Dummy packets define P for the free-running sources
dummy = np.zeros((4, 1, 8), dtype=np.int32)
out, = pseudo(pkt_src=dummy)
print("pseudo-input paced out shape:", out.shape)
print("matches Rom broadcast:", np.allclose(out[:, 0, :], rom_weights))


pseudo-input paced out shape: (4, 1, 8)
matches Rom broadcast: True


## Optional: simulate the Rom+Add graph


In [4]:
rom_model(din=din)
ret = run_flow(rom_model, sim=True)


Dumping input and output data...
Dumping rom weights. Total weights: 8
Running model ROM_ADD_DAG

Running Simulation (vcs)...
Running VCS in sim_ROM_ADD_DAG (device_family=Agilex 7)
*** Using c compiler gcc instead of cc ...
                         Chronologic VCS (TM)
      Version V-2023.12-SP2-1_Full64 -- Fri Sep  4 12:31:07 2026

                    Copyright (c) 1991 - 2024 Synopsys, Inc.
   This software and the associated documentation are proprietary to Synopsys,
 Inc. This software may only be used in accordance with the terms and conditions
 of a written license agreement with Synopsys, Inc. All other use, reproduction,
   or distribution of this software is strictly prohibited.  Licensed Products
     communicate with Synopsys servers for the purpose of providing software
    updates, detecting software piracy and verifying that customers are using
    Licensed Products in conformity with the applicable License Key for such
  Licensed Products. Synopsys will use information

/nfs/site/disks/psg_ctools_1/gcc/13.2.0/1/linux64/rhel7/bin/../lib/gcc/x86_64-none-linux/13.2.0/../../../../x86_64-none-linux/bin/ld: warning: rmar_nd.o: missing .note.GNU-stack section implies executable stack
/nfs/site/disks/psg_ctools_1/gcc/13.2.0/1/linux64/rhel7/bin/../lib/gcc/x86_64-none-linux/13.2.0/../../../../x86_64-none-linux/bin/ld: NOTE: This behaviour is deprecated and will be removed in a future version of the linker


rm -f _cuarc*.so _csrc*.so pre_vcsobj_*.so share_vcsobj_*.so
if [ -x ../simv ]; then chmod a-x ../simv; fi
g++  -o ../simv   -no-pie    -rdynamic  -Wl,-rpath='$ORIGIN'/simv.daidir -Wl,-rpath=./simv.daidir -Wl,-rpath=/p/psg/eda/synopsys/vcsmx/V-2023.12-SP2-1/linux64/suse/suse64/lib -L/p/psg/eda/synopsys/vcsmx/V-2023.12-SP2-1/linux64/suse/suse64/lib  -Wl,-rpath-link=./   objs/amcQw_d.o   _28686_archive_1.so   SIM_l.o       rmapats_mop.o rmapats.o rmar.o rmar_nd.o  rmar_llvm_0_1.o rmar_llvm_0_0.o            -lnuma -lvirsim -lerrorinf -lsnpsmalloc -lvfs    -lvcsnew -lsimprofile -lreader_common /p/psg/eda/synopsys/vcsmx/V-2023.12-SP2-1/linux64/suse/suse64/lib/libBA.a -luclinative /p/psg/eda/synopsys/vcsmx/V-2023.12-SP2-1/linux64/suse/suse64/lib/vcs_tls.o   -Wl,-whole-archive  -lvcsucli    -Wl,-no-whole-archive       ./../simv.daidir/vc_hdrs.o    /p/psg/eda/synopsys/vcsmx/V-2023.12-SP2-1/linux64/suse/suse64/lib/vcs_save_restore_new.o -ldl  -lc -lm -lpthread -ldl 
/nfs/site/disks/psg_ctools_1